# Irish Hospital Demand Forecaster
**CCT College Dublin — Capstone Project 2026**  
Andrei Chistiakov | Ivan Popov

## Notebook Structure
1. Setup & imports
2. Load legacy data (2014–2021)
3. Load OpenData series (2021–2026) — long format
4. Merge into one consistent 12-year series
5. Exploratory Data Analysis
6. COVID-19 effect analysis
7. Feature engineering
8. Model training (XGBoost, Random Forest, Gradient Boosting)
9. Baseline comparison (Naive, Moving Average vs ML)

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_DIR = 'data'
print('Environment ready.')
print(f'Files in data/: {len(os.listdir(DATA_DIR))}')

## 2. Load Legacy Data (2014–2021)

The IPDC-Waiting-List files (2014–2021) use **wide format** — one row per 
hospital/specialty with separate columns for each time band.
We sum across all time bands to get a total count per row.

In [ ]:
def load_legacy_ipdc(data_dir):
    """
    Load IPDC-Waiting-List-By-Group-Hospital files (2014-2021).
    These use wide format with time band columns.
    Returns monthly national totals.
    """
    frames = []
    for fname in sorted(os.listdir(data_dir)):
        if not fname.startswith('IPDC-Waiting-List') or not fname.endswith('.csv'):
            continue
        path = os.path.join(data_dir, fname)
        try:
            df = pd.read_csv(path, encoding='latin-1')
            # Normalise column names
            df.columns = (df.columns
                          .str.replace('ï»¿', '', regex=False)
                          .str.strip().str.lower().str.replace(' ', '_'))

            # Find date column
            date_col = next((c for c in df.columns if 'archive' in c or 'date' in c), None)
            if not date_col:
                continue
            df['archive_date'] = pd.to_datetime(df[date_col], errors='coerce')

            # Wide format: sum all numeric time-band columns
            # These are columns like '0-6_months', '6-12_months', etc.
            # or the Count column if present
            count_col = next((c for c in df.columns if c == 'count'), None)
            if count_col:
                df['row_total'] = pd.to_numeric(df[count_col], errors='coerce').fillna(0)
            else:
                # Sum all numeric columns that aren't ID/date columns
                exclude = {'archive_date', date_col, 'group', 'hospital_hipe',
                           'hospital', 'specialty_hipe', 'specialty',
                           'case_type', 'adult/child', 'age_categorisation',
                           'time_bands', 'adult_child'}
                num_cols = [c for c in df.columns
                            if c not in exclude
                            and pd.api.types.is_numeric_dtype(df[c])]
                if not num_cols:
                    continue
                df['row_total'] = df[num_cols].apply(
                    pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)

            df = df.dropna(subset=['archive_date'])
            df['source'] = 'legacy'
            frames.append(df[['archive_date', 'row_total', 'source']])
            print(f'  Loaded {fname}: {len(df):,} rows')
        except Exception as e:
            print(f'  Skipped {fname}: {e}')

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    # Aggregate to monthly national totals
    monthly = (combined
               .groupby(pd.Grouper(key='archive_date', freq='MS'))['row_total']
               .sum().reset_index()
               .rename(columns={'archive_date': 'year_month', 'row_total': 'waiting_total'}))
    monthly['series'] = 'Legacy IPDC (2014-2021)'
    print(f'\nLegacy monthly snapshots: {len(monthly)}')
    print(f'Range: {monthly["year_month"].min().date()} to {monthly["year_month"].max().date()}')
    return monthly

print('Loading legacy IPDC data (2014-2021)...')
df_legacy = load_legacy_ipdc(DATA_DIR)
df_legacy.tail(3)

## 3. Load OpenData Series (2021–2026)

The OpenData files use **long format** — one row per hospital/specialty/time-band/age combination with a single Count column.
We filter to IPDC files only (matching the legacy series) so the combined dataset is like-for-like.

In [ ]:
def load_opendata_ipdc(data_dir):
    """
    Load OpenData_IPDCNational files (2021-2026).
    """
    frames = []

    for fname in sorted(os.listdir(data_dir)):
        if not fname.startswith('OpenData_IPDC') or not fname.endswith('.csv'):
            continue

        path = os.path.join(data_dir, fname)

        try:
            df = pd.read_csv(path, encoding='latin-1')

            df.columns = (df.columns
                          .str.replace('ï»¿', '', regex=False)
                          .str.strip()
                          .str.lower()
                          .str.replace(' ', '_'))

            date_col  = next((c for c in df.columns if 'archive' in c), None)
            count_col = next((c for c in df.columns if c in ('count', 'total')), None)

            if not date_col or not count_col:
                continue

            df['archive_date'] = pd.to_datetime(df[date_col], errors='coerce')
            df['row_total'] = pd.to_numeric(df[count_col], errors='coerce').fillna(0)

            frames.append(
                df[['archive_date', 'row_total']].dropna()
            )

            print(f'  Loaded {fname}: {len(df):,} rows')

        except Exception as e:
            print(f'  Skipped {fname}: {e}')

    combined = pd.concat(frames, ignore_index=True)

    monthly = (
        combined
        .groupby(pd.Grouper(key='archive_date', freq='MS'))['row_total']
        .sum()
        .reset_index()
        .rename(columns={
            'archive_date': 'year_month',
            'row_total': 'waiting_total'
        })
    )

    monthly['series'] = 'OpenData IPDC (2021-2026)'

    monthly = monthly[
        monthly['waiting_total'] > 10000
    ].reset_index(drop=True)

    print(f'\nOpenData IPDC monthly snapshots: {len(monthly)}')
    print(f'Range: {monthly["year_month"].min().date()} to {monthly["year_month"].max().date()}')

    return monthly


print('Loading OpenData IPDC series (2021-2026)...')
df_opendata = load_opendata_ipdc(DATA_DIR)

## 4. Stack into One Consistent 12-Year Series

The legacy and OpenData series overlap in 2021. We use the OpenData figures for the overlap period
(more granular and consistently formatted) and legacy for 2014–2020.2020.

In [ ]:
# Use legacy for pre-2021, OpenData from 2021 onwards
cutoff = pd.Timestamp('2021-01-01')

legacy_part   = df_legacy[df_legacy['year_month'] < cutoff].copy()
opendata_part = df_opendata.copy()

df_combined = pd.concat([legacy_part, opendata_part], ignore_index=True)
df_combined = df_combined.sort_values('year_month').reset_index(drop=True)

print(f'Combined series: {len(df_combined)} monthly snapshots')
print(f'Range: {df_combined["year_month"].min().date()} to {df_combined["year_month"].max().date()}')
print(f'\nBy source:')
print(df_combined.groupby('series')['waiting_total'].agg(['count','mean','max']).round(0))

In [ ]:
# Visualise the full 12-year series with source highlighted
fig, ax = plt.subplots(figsize=(14, 5))

legacy_plot = df_combined[df_combined['series'].str.startswith('Legacy')]
opendata_plot = df_combined[df_combined['series'].str.startswith('OpenData')]

ax.fill_between(legacy_plot['year_month'],   legacy_plot['waiting_total'],   alpha=0.15, color='#f59e0b')
ax.fill_between(opendata_plot['year_month'], opendata_plot['waiting_total'], alpha=0.15, color='#2171b5')
ax.plot(legacy_plot['year_month'],   legacy_plot['waiting_total'],   color='#f59e0b', linewidth=2, label='Legacy IPDC (2014–2020)')
ax.plot(opendata_plot['year_month'], opendata_plot['waiting_total'], color='#2171b5', linewidth=2, label='OpenData IPDC (2021–2026)')
ax.axvline(x=cutoff, color='grey', linestyle='--', linewidth=1, alpha=0.7, label='Format change')

ax.set_title('NTPF IPDC Waiting List — Full 12-Year Series (2014–2026)', fontsize=13, fontweight='bold')
ax.set_ylabel('Patients Waiting')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('data/full_12year_series.png', dpi=150)
plt.show()
print('Note: legacy files use wide format — summed across time bands.')
print('Both series measure IPDC (Inpatient/Day Case) waiting totals.')

## 5. Exploratory Data Analysis

In [ ]:
# Month-over-month change
df_combined['mom_change'] = df_combined['waiting_total'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df_combined['year_month'], df_combined['waiting_total'], color='#2171b5', linewidth=2)
axes[0].fill_between(df_combined['year_month'], df_combined['waiting_total'], alpha=0.1, color='#2171b5')
axes[0].set_title('National IPDC Waiting List — Monthly Total', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Patients Waiting')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

colours = ['#d73027' if x > 0 else '#1a9641' for x in df_combined['mom_change'].fillna(0)]
axes[1].bar(df_combined['year_month'], df_combined['mom_change'], color=colours, alpha=0.7, width=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Month-over-Month Change (%)', fontsize=12)
axes[1].set_ylabel('% Change')

plt.tight_layout()
plt.savefig('data/eda_trend_mom.png', dpi=150)
plt.show()

In [ ]:
df_combined['cal_month'] = df_combined['year_month'].dt.month
monthly_avg = df_combined.groupby('cal_month')['waiting_total'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(month_names, monthly_avg.values, color='#2171b5', alpha=0.8)

ax.set_title('Average Waiting List by Calendar Month — Seasonality Analysis', 
             fontsize=12, fontweight='bold')
ax.set_ylabel('Average Patients Waiting')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

peak_idx = monthly_avg.values.argmax()
bars[peak_idx].set_color('#d73027')
ax.annotate(f'Peak: {month_names[peak_idx]}',
            xy=(peak_idx, monthly_avg.values[peak_idx]),
            xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('data/eda_seasonality.png', dpi=150)
plt.show()
print(f'Seasonality range: {monthly_avg.min():,.0f} to {monthly_avg.max():,.0f}')
print(f'Variation: {(monthly_avg.max()-monthly_avg.min())/monthly_avg.mean()*100:.1f}% — relatively low seasonal effect')

## 6. COVID-19 Effect Analysis

The data spans the COVID-19 pandemic period. Key events:
- **2020**: Elective procedures suspended during lockdowns → demand suppressed
- **2021**: Rapid return of deferred demand → sharp rise in waiting lists
- **2024**: Peak period before Waiting Time Action Plan intervention
- **2026**: WTAP 2026 shows early effect (Feb–Mar drop)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df_combined['year_month'], df_combined['waiting_total'], alpha=0.12, color='#2171b5')
ax.plot(df_combined['year_month'], df_combined['waiting_total'], color='#2171b5', linewidth=2)

annotations = [
    ('2020-04-01', 82000, 'COVID-19\nlockdowns\n(elective suspended)'),
    ('2021-09-01', 88000, 'Post-COVID\ndemand surge'),
    ('2024-03-01', 95000, 'Pre-WTAP\npeak'),
    ('2026-02-01', 74000, 'WTAP 2026\nearly effect'),
]
for date, y, label in annotations:
    ts = pd.Timestamp(date)
    if df_combined['year_month'].min() <= ts <= df_combined['year_month'].max():
        ax.annotate(label,
                    xy=(ts, df_combined.loc[df_combined['year_month'] == ts, 'waiting_total'].values[0]
                        if ts in df_combined['year_month'].values else y),
                    xytext=(ts, y),
                    ha='center', fontsize=8.5, color='#333',
                    arrowprops=dict(arrowstyle='->', color='#888', lw=1.2))

ax.set_title('IPDC Waiting List with Key Domain Events Annotated', fontsize=12, fontweight='bold')
ax.set_ylabel('Patients Waiting')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('data/annotated_trend.png', dpi=150)
plt.show()

## 7. Feature Engineering

11 features are engineered from the monthly series:
- **Lag features** (1, 3, 6 months): direct autocorrelation signals
- **Rolling stats** (3m, 6m mean and 3m std): momentum and volatility
- **Calendar features** (month sin/cos, year, quarter): cyclical seasonality encoding
- **Trend**: linear counter capturing secular growth direction

In [ ]:
df_model = df_combined[['year_month', 'waiting_total']].copy()

# Time and cyclical features
df_model['month'] = df_model['year_month'].dt.month
df_model['month_sin'] = np.sin(2 * np.pi * df_model['month'] / 12)
df_model['month_cos'] = np.cos(2 * np.pi * df_model['month'] / 12)
df_model['year'] = df_model['year_month'].dt.year
df_model['quarter'] = df_model['year_month'].dt.quarter
df_model['trend'] = range(len(df_model))

# Lag features
df_model['lag_1m'] = df_model['waiting_total'].shift(1)
df_model['lag_3m'] = df_model['waiting_total'].shift(3)
df_model['lag_6m'] = df_model['waiting_total'].shift(6)

# Rolling statistics
df_model['rolling_3m_mean'] = df_model['waiting_total'].rolling(3).mean()
df_model['rolling_6m_mean'] = df_model['waiting_total'].rolling(6).mean()
df_model['rolling_3m_std'] = df_model['waiting_total'].rolling(3).std()

df_features = df_model.dropna().reset_index(drop=True)

FEATURE_COLS = [
    'month_sin', 'month_cos', 'year', 'quarter', 'trend',
    'lag_1m', 'lag_3m', 'lag_6m',
    'rolling_3m_mean', 'rolling_6m_mean', 'rolling_3m_std'
]
TARGET = 'waiting_total'

print(f'Feature matrix: {df_features.shape[0]} rows × {len(FEATURE_COLS)} features')
print(f'Training range: {df_features["year_month"].min().date()} to {df_features["year_month"].max().date()}')

# Correlation with target
corr = df_features[FEATURE_COLS + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)
print(f'\nFeature correlations with target:')
print(corr.round(3).to_string())

## 8. Model Training with TimeSeriesSplit Cross-Validation

**Important:** Standard k-fold cross-validation is NOT used here. k-fold randomly shuffles data,
which would allow training on future months and testing on past months — this is data leakage.

Instead, `TimeSeriesSplit` with 5 folds is used. Each fold's training set ends strictly before
its validation set begins. The last 6 months are held out completely as an unseen test set.

In [ ]:
X = df_features[FEATURE_COLS].values
y = df_features[TARGET].values

# Holdout: last 6 months completely unseen during training
X_train, X_test = X[:-6], X[-6:]
y_train, y_test = y[:-6], y[-6:]

# TimeSeriesSplit CV on training set only
tscv = TimeSeriesSplit(n_splits=5)

models = {
    'XGBoost': xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    'Ridge': Ridge(alpha=1.0),
}

cv_results = []
trained_models = {}

for name, m in models.items():
    fold_maes = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
        m.fit(X_train[tr_idx], y_train[tr_idx])
        preds = m.predict(X_train[val_idx])
        fold_maes.append(mean_absolute_error(y_train[val_idx], preds))
    # Refit on all training data
    m.fit(X_train, y_train)
    trained_models[name] = m
    cv_results.append({
        'Model': name,
        'CV MAE (mean)': f"{np.mean(fold_maes):,.0f}",
        'CV MAE (std)':  f"± {np.std(fold_maes):,.0f}",
    })
    print(f'{name}: CV MAE = {np.mean(fold_maes):,.0f} ± {np.std(fold_maes):,.0f}')

print('\n=== Cross-Validation Results (TimeSeriesSplit, 5 folds) ===')
print(pd.DataFrame(cv_results).to_string(index=False))

# Use XGBoost as primary model
model = trained_models['XGBoost']

## 9. Baseline Model Comparison

Any ML model must beat simple baselines to justify its complexity.
Two baselines are used:
- **Naive forecast**: last known value repeated for all future months
- **Moving average (3m)**: average of last 3 months repeated forward

All models evaluated on the same unseen 6-month holdout.

In [ ]:
# Baseline predictions
last_known = y_train[-1]
ma3_value = y_train[-3:].mean()

naive_preds = np.full(len(y_test), last_known)
ma3_preds = np.full(len(y_test), ma3_value)

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'MAE': f'{mae:,.0f}', 'RMSE': f'{rmse:,.0f}', 
            'MAPE': f'{mape:.1f}%', 'MAE_raw': mae, 'preds': y_pred}

comparison = []
all_preds = {}

for name, m in trained_models.items():
    p = m.predict(X_test)
    r = evaluate(name, y_test, p)
    comparison.append(r)
    all_preds[name] = p

comparison.append(evaluate('Naive (last value)', y_test, naive_preds))
comparison.append(evaluate('Moving Average (3m)', y_test, ma3_preds))

all_preds['Naive'] = naive_preds
all_preds['MA(3)'] = ma3_preds

comp_df = pd.DataFrame(comparison)[['Model','MAE','RMSE','MAPE']]

print('=== Model Comparison — Holdout Oct 2025 to Mar 2026 ===')
print(comp_df.to_string(index=False))

In [ ]:
# Visualise baseline comparison
months = df_features['year_month'].iloc[-6:]
colours_map = {
    'XGBoost':           ('#d73027', '--'),
    'Random Forest':     ('#7b2d8b', '-.'),
    'Gradient Boosting': ('#e6842a', ':'),
    'Naive':             ('#f59e0b', ':'),
    'MA(3)':             ('#059669', '--'),
}

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(months, y_test, color='#2171b5', linewidth=3, marker='o', markersize=6, label='Actual', zorder=5)

mae_vals = {r['Model']: r['MAE_raw'] for r in comparison}
for label, preds_arr in all_preds.items():
    if label in colours_map:
        c, ls = colours_map[label]
        mae_v = mae_vals.get(label, mae_vals.get(label.replace(' (last value)','Naive').replace(' (3m)','MA(3)'), 0))
        ax.plot(months, preds_arr, color=c, linewidth=1.8,
                linestyle=ls, marker='s', markersize=4,
                label=f'{label}  (MAE={mae_vals.get(label,0):,.0f})')

ax.set_title('XGBoost vs All Baselines — Holdout Oct 2025 to Mar 2026', fontsize=12, fontweight='bold')
ax.set_ylabel('Patients Waiting')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('data/baseline_comparison.png', dpi=150)
plt.show()

In [ ]:
# Visualise baseline comparison
months = df_features['year_month'].iloc[-6:]
colours_map = {
    'XGBoost':           ('#d73027', '--'),
    'Random Forest':     ('#7b2d8b', '-.'),
    'Gradient Boosting': ('#e6842a', ':'),
    'Naive':             ('#f59e0b', ':'),
    'MA(3)':             ('#059669', '--'),
}

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(months, y_test, color='#2171b5', linewidth=3, marker='o', markersize=6, label='Actual', zorder=5)

mae_vals = {r['Model']: r['MAE_raw'] for r in comparison}
for label, preds_arr in all_preds.items():
    if label in colours_map:
        c, ls = colours_map[label]
        mae_v = mae_vals.get(label, mae_vals.get(label.replace(' (last value)','Naive').replace(' (3m)','MA(3)'), 0))
        ax.plot(months, preds_arr, color=c, linewidth=1.8,
                linestyle=ls, marker='s', markersize=4,
                label=f'{label}  (MAE={mae_vals.get(label,0):,.0f})')

ax.set_title('XGBoost vs All Baselines — Holdout Oct 2025 to Mar 2026', fontsize=12, fontweight='bold')
ax.set_ylabel('Patients Waiting')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('data/baseline_comparison.png', dpi=150)
plt.show()

In [ ]:
# MAE bar chart — visual comparison
fig, ax = plt.subplots(figsize=(10, 4))
models_list = [r['Model'] for r in comparison]
maes_list   = [r['MAE_raw'] for r in comparison]
bar_colours = ['#d73027' if 'XGBoost' in m else
               '#7b2d8b' if 'Forest' in m else
               '#e6842a' if 'Gradient' in m else
               '#94a3b8' for m in models_list]

bars = ax.barh(models_list[::-1], maes_list[::-1], color=bar_colours[::-1], alpha=0.85)
ax.set_xlabel('Mean Absolute Error (patients)')
ax.set_title('Model Comparison — MAE on Holdout Set', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

for bar, val in zip(bars, maes_list[::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('data/mae_comparison.png', dpi=150)
plt.show()

In [ ]:
# Detailed month-by-month results for XGBoost
xgb_preds = all_preds['XGBoost']
print('=== XGBoost Holdout Results (Oct 2025 – Mar 2026) ===')
for i, (actual, pred) in enumerate(zip(y_test, xgb_preds)):
    month = df_features['year_month'].iloc[-6+i].strftime('%b %Y')
    diff  = actual - pred
    pct   = abs(diff) / actual * 100
    print(f'{month}: actual={actual:>8,.0f}  predicted={pred:>8,.0f}  diff={diff:>+8,.0f}  ({pct:.1f}%)')

mae  = mean_absolute_error(y_test, xgb_preds)
rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
mape = np.mean(np.abs((y_test - xgb_preds) / y_test)) * 100
print(f'\nMAE:  {mae:,.0f} patients')
print(f'RMSE: {rmse:,.0f} patients')
print(f'MAPE: {mape:.1f}%')
print(f'\nNote: Feb-Mar 2026 errors driven by WTAP 2026 policy intervention —')
print('not predictable from historical data alone. This is a known ML limitation.')

In [ ]:
# Retrain on ALL data before forecasting
model.fit(X, y)

lag_buffer = list(df_features['waiting_total'].values)
last       = df_features.iloc[-1]
fc_rows    = []

for step in range(1, 13):
    nd = df_features['year_month'].iloc[-1] + pd.DateOffset(months=step)
    feat = {
        'month_sin':       np.sin(2 * np.pi * nd.month / 12),
        'month_cos':       np.cos(2 * np.pi * nd.month / 12),
        'year':            nd.year,
        'quarter':         nd.quarter,
        'trend':           last['trend'] + step,
        'lag_1m':          lag_buffer[-1],
        'lag_3m':          lag_buffer[-3],
        'lag_6m':          lag_buffer[-6],
        'rolling_3m_mean': np.mean(lag_buffer[-3:]),
        'rolling_6m_mean': np.mean(lag_buffer[-6:]),
        'rolling_3m_std':  np.std(lag_buffer[-3:]),
    }
    yp = float(model.predict(np.array([[feat[c] for c in FEATURE_COLS]]))[0])
    ci = np.std(lag_buffer[-6:]) * (1 + 0.1 * step)
    fc_rows.append({'date': nd, 'forecast': round(yp),
                    'lower': round(max(0, yp - ci)), 'upper': round(yp + ci)})
    lag_buffer.append(yp)

fc = pd.DataFrame(fc_rows)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df_model['year_month'], df_model['waiting_total'], alpha=0.1, color='#2171b5')
ax.plot(df_model['year_month'], df_model['waiting_total'],
        color='#2171b5', linewidth=2, label='Historical (NTPF)')
ax.fill_between(fc['date'], fc['lower'], fc['upper'],
                alpha=0.2, color='#d73027', label='Confidence band (heuristic)')
ax.plot(fc['date'], fc['forecast'],
        color='#d73027', linewidth=2.5, linestyle='--', label='Forecast (XGBoost)')
ax.axhline(y=75000, color='green', linestyle=':', linewidth=1.5,
           label='Sláintecare indicative IPDC target')

ax.set_title('IPDC Waiting List — 12-Month Forecast (Trained on 12-Year Series)', fontsize=13, fontweight='bold')
ax.set_ylabel('Patients Waiting')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend(loc='upper left')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('data/forecast_12m.png', dpi=150)
plt.show()

fc.to_csv('data/forecast_12m.csv', index=False)
print('\n12-Month Forecast:')
print(fc.to_string(index=False))